# LABELS: FAULT EPISODES AND WINDOW LABELLING

* Input: `telemetry_raw_v2.pkl`
* Output:
    * `telemetry_labelled_v2.pkl`
    * `fault_episodes_v2.csv`

Four target windows:
| name | seconds before onset | seconds after onset | kind |
|---|---|---|---|
| D5   | 5   | 5  | detection |
| W5   | 5   | −1 | warning   |
| W60  | 60  | −1 | warning   |
| W300 | 300 | −1 | warning   |

* An "after" value of −1 makes a warning window stop one second before onset, so it never contains the onset second. D5 is labelled exactly as W5 was in LABELS; with legacy pairing its counts must reproduce 6,161 / 198,060 / 1,945,552.
* Episode seconds (onset to recovery) are excluded (−1). For D5 the window takes precedence, as before. For the warning windows the episode takes precedence: a second that lies inside any fault episode, including an earlier one still
active, is never labelled as a warning of the next onset. The notebook reports how many window seconds this removes.
* Episode pairing corrected: In LABELS, an onset whose own recovery is missing from the log takes the recovery of the next onset of the same fault, and every later episode of that fault then shifts by one. Durations are inflated, and the inflated episodes exclude seconds (including other onsets' warning windows) that should not be excluded. The corrected rule accepts a
recovery only if it falls before the next onset of the same fault in the same cluster; otherwise the onset is left unmatched. `pairing="legacy"` reproduces LABELS exactly and is used for the D5 reproduction check.
* Onset times, and therefore every count, the daily series and the entropy inputs, are unaffected. Durations, recovery times and the exclusion mask change.

In [ ]:
import os
import numpy as np, pandas as pd

CFG = dict(
    telemetry   = "TELEMETRY/build/telemetry_raw_v2.pkl",
    fault_log   = "DATA/37011FLv1.csv",
    out_labels  = "TELEMETRY/build/telemetry_labelled_v2.pkl",
    out_eps     = "TELEMETRY/build/fault_episodes_v2.csv",
    prev_eps    = "TELEMETRY/build/fault_episodes.csv",
    study_start = "2025-06-01 00:00:00",
    study_end   = "2025-06-30 23:59:59",
    pairing     = "bounded",
    max_episode_s = 6 * 3600,
    debounce_s    = 60,
    horizons = {            # name -> (seconds before, seconds after, kind)
        "D5":   (5,   5,  "detection"),
        "W5":   (5,   -1, "warning"),
        "W60":  (60,  -1, "warning"),
        "W300": (300, -1, "warning"),
    },
)

In [ ]:
# 1. FAULT LOG EPISODES
def build_episodes(cfg, pairing):
    fl = pd.read_csv(cfg["fault_log"])
    fl["dt"] = pd.to_datetime(fl["DATETIME"], format="%m/%d/%Y %H:%M:%S")
    fl["is_rec"] = fl["FAULT"].str.contains("Recovered", case=False, na=False)
    fl["base"] = (fl["FAULT"].str.replace(" Recovered", "", regex=False).str.strip().str.upper())
    fl = fl.sort_values("dt").reset_index(drop=True)

    episodes = []
    for (cluster, base), grp in fl.groupby(["CLUSTER", "base"]):
        onsets = grp.loc[~grp.is_rec, "dt"].tolist()
        recovs = grp.loc[grp.is_rec, "dt"].tolist()
        ri = 0
        for k, on in enumerate(onsets):
            nxt = onsets[k + 1] if k + 1 < len(onsets) else pd.Timestamp.max
            if pairing == "legacy":
                while ri < len(recovs) and recovs[ri] <= on:
                    ri += 1
            else:
                while ri < len(recovs) and recovs[ri] < on:
                    ri += 1
            end, dur, matched = on, 0.0, False
            if ri < len(recovs):
                cand = recovs[ri]
                d = (cand - on).total_seconds()
                ok = d <= cfg["max_episode_s"]
                if pairing != "legacy":
                    ok = ok and cand <= nxt
                if ok:
                    end, dur, matched = cand, d, True
                    ri += 1
            episodes.append(dict(cluster=cluster, fault=base, onset=on, recovery=end, duration_s=dur, matched=matched))

    ep = pd.DataFrame(episodes).sort_values("onset").reset_index(drop=True)
    ep["debounce_keep"] = True
    for c, g in ep.groupby("cluster"):
        gaps = g["onset"].diff().dt.total_seconds()
        ep.loc[g.index, "debounce_keep"] = gaps.isna() | (gaps > cfg["debounce_s"])
    return ep

In [ ]:
# 2. EPISODES: PER-SECOND LABELS
ONE_S = np.timedelta64(1, "s")

def label(index, ep, pre_s, post_s, kind):
    """1 = window, 0 = negative, -1 = excluded (inside a fault episode).
    Returns the labels and the number of window seconds that fall inside an
    episode (kept as positive for detection, excluded for warning)."""
    n = len(index)
    win = np.zeros(n, dtype=bool)
    epi = np.zeros(n, dtype=bool)
    lo = index.searchsorted
    for on, rec in zip(ep["onset"].values, ep["recovery"].values):
        win[lo(on - pre_s * ONE_S, "left"):lo(on + post_s * ONE_S, "right")] = True
        epi[lo(on, "left"):lo(max(rec, on), "right")] = True
    y = np.zeros(n, dtype=np.int8)
    if kind == "detection":
        y[epi] = -1
        y[win] = 1
    else:
        y[win] = 1
        y[epi] = -1
    return y, int((win & epi).sum())

def june(ep, cfg):
    return ep[(ep.onset >= cfg["study_start"]) & (ep.onset <= cfg["study_end"])]

def describe_pairing(ep, name):
    m = ep[ep.matched]
    return dict(pairing=name, episodes=len(ep), matched=int(ep.matched.sum()),
                matched_pct=round(100 * ep.matched.mean(), 1),
                dur_median_s=m.duration_s.median(), dur_mean_s=round(m.duration_s.mean(), 1),
                dur_p90_s=m.duration_s.quantile(0.9), dur_max_s=m.duration_s.max(),
                episode_seconds=int(m.duration_s.sum()))

def build(cfg):
    tel = pd.read_pickle(cfg["telemetry"])
    tel = tel[(tel.DATETIME >= cfg["study_start"]) & (tel.DATETIME <= cfg["study_end"])].copy()
    tel = tel.sort_values("DATETIME").reset_index(drop=True)
    idx = tel["DATETIME"].values

    ep_legacy = build_episodes(cfg, "legacy")
    ep = build_episodes(cfg, cfg["pairing"])
    span, span_legacy = june(ep, cfg), june(ep_legacy, cfg)

    comp = pd.DataFrame([describe_pairing(ep_legacy, "legacy, full log"),
                         describe_pairing(ep, f"{cfg['pairing']}, full log"),
                         describe_pairing(span_legacy, "legacy, June"),
                         describe_pairing(span, f"{cfg['pairing']}, June")]).set_index("pairing")
    key = ["cluster", "fault", "onset"]
    a = ep_legacy.set_index(key)["recovery"]
    b = ep.set_index(key)["recovery"].reindex(a.index)
    print("PAIRING, legacy against corrected (N-15)")
    print(comp.T.to_string())
    print(f"  episodes whose recovery changes: {int((a != b).sum()):,} of {len(a):,}\n")

    print(f"Telemetry seconds in June : {len(tel):,}  (expected 2,149,773)")
    print(f"Duty cycle                : {len(tel)/2_592_000*100:.1f}% of the month\n")
    print("FAULT EPISODES")
    print(f"  episodes, full log      : {len(ep):,}  (expected 4,521)")
    print(f"  episodes, June onsets   : {len(span):,}  (expected 605)")
    print(f"  with matched recovery   : {int(span.matched.sum()):,} "
          f"({span.matched.mean()*100:.1f}%)")
    d = span.loc[span.matched, "duration_s"]
    print(f"  duration: median {d.median():.0f}s  mean {d.mean():.0f}s  "
          f"p90 {d.quantile(.9):.0f}s  max {d.max():.0f}s\n")

    y_ref, _ = label(idx, span_legacy, 5, 5, "detection")
    ref = [int((y_ref == 1).sum()), int((y_ref == -1).sum()), int((y_ref == 0).sum())]

    rows = []
    for name, (pre, post, kind) in cfg["horizons"].items():
        y, overlap = label(idx, span, pre, post, kind)
        tel[name] = y
        n1, nx, n0 = int((y == 1).sum()), int((y == -1).sum()), int((y == 0).sum())
        rows.append(dict(label=name, kind=kind, before_s=pre, after_s=post,
                         positive=n1, pos_pct=round(n1 / (n1 + n0) * 100, 3),
                         excluded=nx, negative=n0,
                         window_s_inside_episode=overlap,
                         treatment="kept positive" if kind == "detection" else "excluded"))
    summary = pd.DataFrame(rows).set_index("label")
    print("WINDOW LABELS")
    print(summary.to_string())
    comp.to_csv("OUTPUT/episode_pairing_v2.csv")
    return tel, ep, span, summary, ref

os.makedirs("OUTPUT", exist_ok=True)
tel, ep, span, summary, ref = build(CFG)

In [ ]:
# 3. CONSISTENCY CHECKS
print("D5 with legacy pairing reproduces LABELS W5:",
      "OK" if ref == [6161, 198060, 1945552] else f"MISMATCH {ref}")
print("D5 with corrected pairing:",
      summary.loc["D5", ["positive", "excluded", "negative"]].tolist())

if os.path.exists(CFG["prev_eps"]):
    prev = pd.read_csv(CFG["prev_eps"], parse_dates=["onset", "recovery"])
    cols = ["cluster", "fault", "onset", "recovery", "matched"]
    same_on = (len(prev) == len(ep) and
               prev["onset"].reset_index(drop=True).equals(ep["onset"].reset_index(drop=True)))
    print("Onsets identical to fault_episodes.csv:", "OK" if same_on else "DIFFERENT")
    moved = (prev["recovery"].values != ep["recovery"].values).sum() if same_on else "n/a"
    print("Recoveries that differ from fault_episodes.csv:", moved)

In [ ]:
# 4. SAVES
tel.to_pickle(CFG["out_labels"])
ep.to_csv(CFG["out_eps"], index=False)
summary.to_csv("OUTPUT/label_summary_v2.csv")
print("written:", CFG["out_labels"], CFG["out_eps"], "OUTPUT/label_summary_v2.csv")

written: TELEMETRY/build/telemetry_labelled_v2.pkl TELEMETRY/build/fault_episodes_v2.csv OUTPUT/label_summary_v2.csv


___
**END**